# Chest X-ray — SMALL run (C3 go/no-go)

**Purpose:** train 3 diverse models on REAL data, run the full pipeline, and read the **§8b C3 verdict** (does SRC predict cross-source collapse?). GO -> use colab_full.ipynb.

Same code as the full run, just 3 models / 20 epochs. One entrypoint: `scripts/run_pipeline.py`.

## Cell 1 — mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — clone code from GitHub

In [ ]:
import os, subprocess
REPO = 'https://github.com/kamlishgoswami/Chest_Xray.git'
if not os.path.exists('/content/Chest_Xray'):
    subprocess.run(['git','clone', REPO, '/content/Chest_Xray'], check=True)
else:
    subprocess.run(['git','-C','/content/Chest_Xray','pull'], check=True)
os.chdir('/content/Chest_Xray'); print('cwd:', os.getcwd())

## Cell 3 — unzip data to fast local disk (slow, once per session)

In [ ]:
import os, glob, subprocess
DRIVE_DIR = '/content/drive/MyDrive/cxr_data'
os.makedirs('data/raw', exist_ok=True)
for z in glob.glob(f'{DRIVE_DIR}/*.zip'):
    print('unzipping', os.path.basename(z))
    subprocess.run(['unzip','-q','-o', z, '-d','data/raw'], check=True)
print('folders:', os.listdir('data/raw'))

## Cell 4 — bring the manifest

In [ ]:
import os, shutil, subprocess
DRIVE_DIR = '/content/drive/MyDrive/cxr_data'
if os.path.exists(f'{DRIVE_DIR}/manifest.csv'):
    shutil.copy(f'{DRIVE_DIR}/manifest.csv','data/manifest.csv'); print('manifest copied')
else:
    subprocess.run(['python','scripts/build_manifest.py'], check=True); print('rebuilt')

## Cell 5 — install deps + GPU check (GPU must NOT be empty)

In [ ]:
import subprocess
subprocess.run(['pip','-q','install','-r','requirements.txt'], check=True)
import tensorflow as tf
print('GPU:', tf.config.list_physical_devices('GPU'))

## Cell 6 — DATA PATH CHECK ⛔ (stop if not 0)

In [ ]:
import pandas as pd, os
df = pd.read_csv('data/manifest.csv')
missing = [p for p in df['image_path'].head(300) if not os.path.exists(p)]
print('missing:', len(missing), '/ 300')
if missing:
    print('EXAMPLE:', missing[0]); print('folders:', os.listdir('data/raw'))
    print('>>> STOP: fix folder mapping before running.')
else:
    print('>>> OK: paths resolve. Continue.')

## Cell 7 — FULL pipeline on 3 models (the slow cell, ~20-40 min)
Runs P1..P9: train -> eval -> CSA/SRC -> cross-source+C3 -> XAI -> robust -> abstain -> stats -> report.

In [ ]:
import subprocess, sys
MODELS = ['densenet201','resnet50','vit']
subprocess.run([sys.executable,'scripts/run_pipeline.py',
                '--models',*MODELS,'--epochs','20','--batch-size','64'], check=True)

## Cell 8 — READ THE §8b VERDICT (decide threshold BEFORE looking)

In [ ]:
import json
c = json.load(open('results/c3_coupling.json'))
print(json.dumps(c, indent=2))
if c.get('n_models',0) >= 3 and 'delta_acc' in c:
    r2 = c['delta_acc']['r2']; r = c['delta_acc']['r']
    print(f"\nSRC -> delta_acc:  r={r:+.3f}  R2={r2:.3f}")
    print('GO -> colab_full.ipynb (spine = METHOD)' if (r>0 and r2>=0.3)
          else 'WEAK/NO-GO -> spine = AUDIT; rethink SRC (PAPER_OUTLINE.md §8b)')
else:
    print('Need 3 models with valid SRC + cross-source results.')

## Cell 9 — SAVE results to Drive (Colab wipes /content at session end)

In [ ]:
import subprocess, datetime
stamp = datetime.date.today().isoformat()
subprocess.run(['cp','-r','results',f'/content/drive/MyDrive/cxr_data/results_small_{stamp}'], check=True)
print('saved -> Drive/cxr_data/results_small_'+stamp)